# Ingesta de CSV a Milvus para RAG
Este notebook demuestra cómo procesar los datos estructurados para optimizar la búsqueda vectorial con filtros de metadatos (Hybrid Search).

In [1]:
!pip install pandas langchain-core langchain-milvus langchain_openai "pymilvus>=2.5.0,<2.6.0"

INFO: pip is looking at multiple versions of langchain-milvus to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 74.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 363.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 231.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 593.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 351.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 461.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 607.7 MB/s  0:00:00
  Attempting uninstall: idnam━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/21 [regex]
    Found existing installation: idna 3.11━━━━━━━━━━━━━━━━━━━━  4/21 [regex]
    Uninstalling idna-3.11:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/21 [regex]
      Successfully uninstalled idna-3.11━━━━━━━━━━━━━━━━━━━━━━  4/21 [regex]
  

In [ ]:
import pandas as pd
from langchain_core.documents import Document
from langchain_milvus import Milvus
import requests
from typing import List
from langchain_core.embeddings import Embeddings
from pymilvus import MilvusClient
import json

In [29]:
file_path = "data/Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv"
url = "https://www.openshift.guide/openshift-guide-screen.pdf"
vector_db_skus_name = "skus_rh_vector_db"
vector_db_ocp_name = "ocp_rh_vector_db"
milvus_uri = "http://milvus-service.proposal-rh-ai.svc.cluster.local:19530"

milvus_client = MilvusClient(uri=milvus_uri)

print("Connected to MilvusClient")

Connected to MilvusClient


In [30]:
print("Procesando vulnerabilidades y mitigaciones...")
df = pd.read_csv(file_path)
processed_skus = []

for index, row in df.iterrows():
    sku = str(row.get('SKU', ''))
    sku_description = str(row.get('SKU_Description', ''))

    contenido_semantico = f"""
    Producto: {row['Product']} ({row['Category']}) - SKU: {sku}.
    Descripción: {row['SKU_Description']}.
    Precio: {row['List_Price']} {row['Currency']} por {row['Unit_of_Measure']}.
    Soporte: Nivel {row['Support_Level']}, Tipo {row['Support_Type']} (Término: {row['Service_Term']}).
    Especificaciones: {row['Cores']} Cores, {row['Nodes']} Nodes, {row['Sockets']} Sockets, {row['Virtual_Guests']} Virtual Guests.
    Disponibilidad: Región {row['Region']}, {row['Country']} ({row['YEAR']} {row['QUARTER']}).
    """

    doc = Document(
        page_content=contenido_semantico,
        metadata={
            "source": "Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv",
            "sku": sku,
            "name": sku_description
        }
    )
    processed_skus.append(doc)

print(f"¡Listo! Se procesaron {len(processed_skus)} SKUs.")

Procesando vulnerabilidades y mitigaciones...
¡Listo! Se procesaron 680 SKUs.


In [31]:
if milvus_client.has_collection(collection_name=vector_db_skus_name):
    milvus_client.drop_collection(collection_name=vector_db_skus_name)
    print(f"Colección '{vector_db_skus_name}' eliminada correctamente.")

Colección 'skus_rh_vector_db' eliminada correctamente.


In [32]:
class LlamaStackGraniteEmbeddings(Embeddings):
    """
    Adaptador personalizado para enviar textos puros a Llama Stack,
    evitando los bugs de arrays del SDK de OpenAI.
    """

    def __init__(self, base_url: str, model: str):
        self.base_url = base_url
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        vectores = []
        # Enviamos los textos uno por uno como strings puros
        for text in texts:
            # Aseguramos que sea un string válido
            texto_limpio = str(text).strip()
            payload = {
                "model": self.model,
                "input": texto_limpio
            }
            response = requests.post(
                f"{self.base_url}/embeddings",  # Endpoint /v1/embeddings
                json=payload,
                headers={"Content-Type": "application/json"}
            )
            # Si hay un error HTTP, nos lo mostrará claramente
            response.raise_for_status()
            # Extraemos el vector de la respuesta de Llama Stack
            vector = response.json()["data"][0]["embedding"]
            vectores.append(vector)
        return vectores

    def embed_query(self, text: str) -> List[float]:
        # Para LangChain, un query es solo un documento de 1 elemento
        return self.embed_documents([text])[0]


embeddings = LlamaStackGraniteEmbeddings(
    model="sentence-transformers/ibm-granite/granite-embedding-125m-english", 
    base_url="http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1"
)

In [33]:
print("Conectando a Milvus")

vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": milvus_uri},
    collection_name=vector_db_skus_name,
    index_params={"index_type": "FLAT", "metric_type": "COSINE"},
    auto_id=True,
    enable_dynamic_field=True,
)

print("Ingestando vectores de SKU documents en Milvus mediante LangChain...")

vectorstore_cwe = Milvus.from_documents(
    documents=processed_skus,
    embedding=embeddings,
    connection_args={"uri": milvus_uri},
    collection_name=vector_db_skus_name,
    drop_old=True,
    auto_id=True,
    enable_dynamic_field=True,
    index_params={"index_type": "FLAT", "metric_type": "COSINE"},
)
print("✅ ¡Ingesta de SKUs documents completada con éxito!")

Conectando a Milvus
Ingestando vectores de SKU documents en Milvus mediante LangChain...
✅ ¡Ingesta de SKUs documents completada con éxito!


In [44]:
question = "Puedes listar los SKUs del producto Advanced Cluster Management que conozcas, solo toma la información del RAG que tienes disponible y detalla toda la información que tengas"

search_res = milvus_client.search(
    collection_name=vector_db_skus_name,
    data=[
        embeddings.embed_query(question)
    ],
    search_params={"metric_type": "COSINE", "params": {}},
    output_fields=["text"],
)

chunks = []
for hit in search_res[0]:
    texto = hit["entity"].get("text", "")
    chunks.append(f"[Doc SKU]: {texto}")

print(f"Chunks: {chunks}")

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

Chunks: ['[Doc SKU]: \n    Producto: ACM - Advanced Cluster Management (SUBSCRIPTIONS) - SKU: MW04497.\n    Descripción: Red Hat Advanced Cluster Management for Kubernetes (Bare Metal Node), Standard (1-2 Sockets up to 128 Cores).\n    Precio: 6.600,00 USD por PHYSICAL NODE.\n    Soporte: Nivel L1-L3, Tipo Standard (Término: 1 YEARS).\n    Especificaciones: 0 Cores, 0 Nodes, 2 Sockets, 0 Virtual Guests.\n    Disponibilidad: Región LATAM, ALL (2025 Q3).\n    ', '[Doc SKU]: \n    Producto: ACM - Advanced Cluster Management (SUBSCRIPTIONS) - SKU: MW02614.\n    Descripción: Red Hat Advanced Cluster Management for Kubernetes for IBM Z and IBM LinuxONE, Premium (1 Core).\n    Precio: 1.650,00 USD por IFL.\n    Soporte: Nivel L1-L3, Tipo Premium (Término: 1 YEARS).\n    Especificaciones: 1 Cores, 0 Nodes, 0 Sockets, 0 Virtual Guests.\n    Disponibilidad: Región LATAM, ALL (2025 Q3).\n    ', '[Doc SKU]: \n    Producto: ACM - Advanced Cluster Management (SUBSCRIPTIONS) - SKU: MW04496.\n    Desc